In [1]:
%%bash
set -euxo pipefail

sudo apt-get update -y
sudo apt-get install -y \
  python3.10 python3.10-venv python3.10-dev \
  git ninja-build build-essential \
  libgl1 libglib2.0-0 ffmpeg curl

curl -LsSf https://astral.sh/uv/install.sh | sh
export PATH="$HOME/.local/bin:$PATH"

uv --version
python3.10 --version

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [87.4 kB]
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,454 kB]
Get:6 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:7 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:8 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,901 kB]
Hit:13 https://ppa.launchpadcontent.net/graphics-dr

+ sudo apt-get update -y
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
+ sudo apt-get install -y python3.10 python3.10-venv python3.10-dev git ninja-build build-essential libgl1 libglib2.0-0 ffmpeg curl
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 15.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
+ curl -LsSf https://astral.sh/uv/install.sh
+ sh
downloading uv 0.10.12 x86_64-unknown-linux-gnu
no checksums to verify
+ export PATH=/root/.local/bin:/opt/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/lo

In [2]:
%%bash
set -euxo pipefail
export PATH="$HOME/.local/bin:$PATH"

cd /content
rm -rf /content/TRELLIS /content/py310

git clone --recurse-submodules https://github.com/DopeMeerkat/TRELLIS.git
cd /content/TRELLIS
git submodule update --init --recursive

Submodule path 'trellis/representations/mesh/flexicubes': checked out '815e075a2a400d06c48d94c347674344ed6ae5c5'


+ export PATH=/root/.local/bin:/opt/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tools/node/bin:/tools/google-cloud-sdk/bin
+ PATH=/root/.local/bin:/opt/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tools/node/bin:/tools/google-cloud-sdk/bin
+ cd /content
+ rm -rf /content/TRELLIS /content/py310
+ git clone --recurse-submodules https://github.com/DopeMeerkat/TRELLIS.git
Cloning into 'TRELLIS'...
Submodule 'trellis/representations/mesh/flexicubes' (https://github.com/MaxtirError/FlexiCubes.git) registered for path 'trellis/representations/mesh/flexicubes'
Cloning into '/content/TRELLIS/trellis/representations/mesh/flexicubes'...
+ cd /content/TRELLIS
+ git submodule update --init --recursive


In [4]:
%%bash
set -euxo pipefail
export PATH="$HOME/.local/bin:$PATH"

uv venv --python 3.10 /content/py310
source /content/py310/bin/activate

cd /content/TRELLIS
uv sync --active

+ export PATH=/root/.local/bin:/opt/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tools/node/bin:/tools/google-cloud-sdk/bin
+ PATH=/root/.local/bin:/opt/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tools/node/bin:/tools/google-cloud-sdk/bin
+ uv venv --python 3.10 /content/py310
Using CPython 3.10.12 interpreter at: /usr/bin/python3.10
Creating virtual environment at: py310
Activate with: source py310/bin/activate
+ source /content/py310/bin/activate
++ '[' -z '' ']'
++ '[' -n x ']'
++ SCRIPT_PATH=/content/py310/bin/activate
++ '[' /content/py310/bin/activate = bash ']'
++ deactivate nondestructive
++ unset -f pydoc
++ '[' -z '' ']'
++ '[' -z '' ']'
++ hash -r
++ '[' -z '' ']'
++ unset VIRTUAL_ENV
++ unset VIRTUAL_ENV_PROMPT
++ '[' '!' nondestructive = nondestructive ']'
++ VIRTUAL_ENV=/content/py310
++ '[' linux-gnu = cygwin ']'
++ '[' linux-gnu = msys ']'
++ export VIRTUAL_ENV
++ '[' -z '' ']'
++ unset 

In [5]:
%%bash
set -euxo pipefail
export PATH="$HOME/.local/bin:$PATH"

source /content/py310/bin/activate
cd /content/TRELLIS

rm -rf /tmp/extensions || true

export CUDA_HOME=/usr/local/cuda
export PATH="$CUDA_HOME/bin:$PATH"
export TORCH_CUDA_ARCH_LIST="7.5"

uv pip uninstall --python /content/py310/bin/python -y flash-attn || true
uv pip install --python /content/py310/bin/python --no-build-isolation -v flash-attn==2.8.3

uv pip install --python /content/py310/bin/python --no-build-isolation \
  "git+https://github.com/NVlabs/nvdiffrast.git@v0.3.4"

python - <<'PY'
import importlib.metadata as md
import os
import torch

print("torch", torch.__version__)
print("cuda", torch.version.cuda)
print("avail", torch.cuda.is_available())
print("gpu", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

import kaolin
print("kaolin", kaolin.__version__)

import flash_attn
print("flash_attn", flash_attn.__version__)

import spconv
print("spconv OK")

import nvdiffrast
print("nvdiffrast", md.version("nvdiffrast"), os.path.dirname(nvdiffrast.__file__))
PY

torch 2.5.1+cu121
cuda 12.1
avail True
gpu NVIDIA A100-SXM4-40GB
kaolin 0.18.0
flash_attn 2.8.3
spconv OK
nvdiffrast 0.3.4 /content/py310/lib/python3.10/site-packages/nvdiffrast


+ export PATH=/root/.local/bin:/opt/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tools/node/bin:/tools/google-cloud-sdk/bin
+ PATH=/root/.local/bin:/opt/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tools/node/bin:/tools/google-cloud-sdk/bin
+ source /content/py310/bin/activate
++ '[' -z '' ']'
++ '[' -n x ']'
++ SCRIPT_PATH=/content/py310/bin/activate
++ '[' /content/py310/bin/activate = bash ']'
++ deactivate nondestructive
++ unset -f pydoc
++ '[' -z '' ']'
++ '[' -z '' ']'
++ hash -r
++ '[' -z '' ']'
++ unset VIRTUAL_ENV
++ unset VIRTUAL_ENV_PROMPT
++ '[' '!' nondestructive = nondestructive ']'
++ VIRTUAL_ENV=/content/py310
++ '[' linux-gnu = cygwin ']'
++ '[' linux-gnu = msys ']'
++ export VIRTUAL_ENV
++ '[' -z '' ']'
++ unset SCRIPT_PATH
++ _OLD_VIRTUAL_PATH=/root/.local/bin:/opt/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tools/node/bin:/tools/google-cloud-s

In [6]:
%%bash
set -euxo pipefail
source /content/py310/bin/activate

python - <<'PY'
from pathlib import Path
import re

p = Path("/content/TRELLIS/trellis/pipelines/base.py")
s = p.read_text()

if not re.search(r"^\s*import os\s*$", s, re.M):
    s = re.sub(r"(?m)^((?:from .+\n|import .+\n)+)", r"\1import os\n", s, count=1)

needle = 'config_file = hf_hub_download(path, "pipeline.json")'
m = re.search(r"(?m)^(?P<indent>\s*)" + re.escape(needle) + r"\s*$", s)
if not m:
    raise SystemExit("Expected line not found: " + needle)

indent = m.group("indent")
replacement = "\n".join([
    f"{indent}# Local folder support: if `path` is a directory, read pipeline.json from disk",
    f"{indent}if os.path.isdir(path):",
    f"{indent}    config_file = os.path.join(path, 'pipeline.json')",
    f"{indent}    if not os.path.exists(config_file):",
    f"{indent}        raise FileNotFoundError('Missing pipeline.json in local folder: ' + str(path))",
    f"{indent}else:",
    f"{indent}    config_file = hf_hub_download(path, 'pipeline.json')",
])

s = re.sub(r"(?m)^" + re.escape(indent) + re.escape(needle) + r"\s*$", replacement, s, count=1)
p.write_text(s)
print("Patched:", p)
PY

Patched: /content/TRELLIS/trellis/pipelines/base.py


+ source /content/py310/bin/activate
++ '[' -z '' ']'
++ '[' -n x ']'
++ SCRIPT_PATH=/content/py310/bin/activate
++ '[' /content/py310/bin/activate = bash ']'
++ deactivate nondestructive
++ unset -f pydoc
++ '[' -z '' ']'
++ '[' -z '' ']'
++ hash -r
++ '[' -z '' ']'
++ unset VIRTUAL_ENV
++ unset VIRTUAL_ENV_PROMPT
++ '[' '!' nondestructive = nondestructive ']'
++ VIRTUAL_ENV=/content/py310
++ '[' linux-gnu = cygwin ']'
++ '[' linux-gnu = msys ']'
++ export VIRTUAL_ENV
++ '[' -z '' ']'
++ unset SCRIPT_PATH
++ _OLD_VIRTUAL_PATH=/opt/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tools/node/bin:/tools/google-cloud-sdk/bin
++ PATH=/content/py310/bin:/opt/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tools/node/bin:/tools/google-cloud-sdk/bin
++ export PATH
++ '[' x '!=' x ']'
+++ basename /content/py310
++ VIRTUAL_ENV_PROMPT=py310
++ export VIRTUAL_ENV_PROMPT
++ '[' -z '' ']'
++ '[' -z '' ']'
++ _OLD_VIRTUAL_PS

In [7]:
%%bash
set -euxo pipefail
source /content/py310/bin/activate

python - <<'PY'
from huggingface_hub import snapshot_download
path = snapshot_download(
    repo_id="microsoft/TRELLIS-image-large",
    local_dir="/content/TRELLIS-image-large",
    local_dir_use_symlinks=False,
)
print("downloaded to:", path)
PY

sed -i 's|from_pretrained("./TRELLIS-image-large")|from_pretrained("/content/TRELLIS-image-large")|g' /content/TRELLIS/generate.py

downloaded to: /content/TRELLIS-image-large


+ source /content/py310/bin/activate
++ '[' -z '' ']'
++ '[' -n x ']'
++ SCRIPT_PATH=/content/py310/bin/activate
++ '[' /content/py310/bin/activate = bash ']'
++ deactivate nondestructive
++ unset -f pydoc
++ '[' -z '' ']'
++ '[' -z '' ']'
++ hash -r
++ '[' -z '' ']'
++ unset VIRTUAL_ENV
++ unset VIRTUAL_ENV_PROMPT
++ '[' '!' nondestructive = nondestructive ']'
++ VIRTUAL_ENV=/content/py310
++ '[' linux-gnu = cygwin ']'
++ '[' linux-gnu = msys ']'
++ export VIRTUAL_ENV
++ '[' -z '' ']'
++ unset SCRIPT_PATH
++ _OLD_VIRTUAL_PATH=/opt/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tools/node/bin:/tools/google-cloud-sdk/bin
++ PATH=/content/py310/bin:/opt/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tools/node/bin:/tools/google-cloud-sdk/bin
++ export PATH
++ '[' x '!=' x ']'
+++ basename /content/py310
++ VIRTUAL_ENV_PROMPT=py310
++ export VIRTUAL_ENV_PROMPT
++ '[' -z '' ']'
++ '[' -z '' ']'
++ _OLD_VIRTUAL_PS

In [8]:
%%bash
set -euxo pipefail
mkdir -p /content/outputs
/content/py310/bin/python /content/TRELLIS/generate.py /content/hambaga.jpg /content/outputs/hambaga

[SPARSE] Backend: spconv, Attention: flash_attn
[SPARSE][CONV] spconv algo: native
[ATTENTION] Using backend: flash_attn
3D generation completed successfully!
Generated 1 Gaussian splat(s)
Generated 1 radiance field(s)
Generated 1 mesh(es)
Saved raw mesh OBJ: /content/outputs/hambaga_raw.obj
Done. Note: Rendering videos is omitted to keep the script lightweight for Colab.


+ mkdir -p /content/outputs
+ /content/py310/bin/python /content/TRELLIS/generate.py /content/hambaga.jpg /content/outputs/hambaga
/content/py310/lib/python3.10/site-packages/spconv/pytorch/functional.py:47: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  _TORCH_CUSTOM_FWD = amp.custom_fwd(cast_inputs=torch.float16)
/content/py310/lib/python3.10/site-packages/spconv/pytorch/functional.py:97: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  def backward(ctx, grad_output):
/content/py310/lib/python3.10/site-packages/spconv/pytorch/functional.py:163: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  def backward(ctx, grad_output):
/content/py310/lib/python3.10/site-packages/spconv/pytorch/functional.py:243: FutureWarn

# NGrok API


In [10]:
%%bash
set -euxo pipefail

wget -q -O /tmp/ngrok.tgz https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz
tar -xzf /tmp/ngrok.tgz -C /tmp

sudo mv /tmp/ngrok /usr/local/bin/ngrok
sudo chmod +x /usr/local/bin/ngrok

ngrok version
which ngrok

ngrok version 3.37.2
/usr/local/bin/ngrok


+ wget -q -O /tmp/ngrok.tgz https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz
+ tar -xzf /tmp/ngrok.tgz -C /tmp
+ sudo mv /tmp/ngrok /usr/local/bin/ngrok
+ sudo chmod +x /usr/local/bin/ngrok
+ ngrok version
+ which ngrok


In [11]:
%%bash
set -euxo pipefail
source /content/py310/bin/activate

export TRELLIS_MODEL_DIR="${TRELLIS_MODEL_DIR:-/content/TRELLIS-image-large}"
export PYTHONPATH="/content/TRELLIS:${PYTHONPATH:-}"

PID_ON_8000="$(ss -ltnp | awk '/:8000/ {print $NF}' | sed -n 's/.*pid=\([0-9]\+\).*/\1/p' | head -n 1 || true)"
if [ -n "$PID_ON_8000" ]; then kill -9 "$PID_ON_8000" || true; fi
rm -rf /content/__pycache__ || true

nohup /content/py310/bin/python -m uvicorn trellis_api_server:app \
  --app-dir /content \
  --host 0.0.0.0 \
  --port 8000 \
  --log-level info \
  > /content/uvicorn.log 2>&1 &

echo $! > /content/uvicorn.pid

for i in $(seq 1 60); do
  if curl -fsS http://127.0.0.1:8000/health >/dev/null 2>&1; then
    curl -fsS http://127.0.0.1:8000/health
    echo
    echo "Server up. PID=$(cat /content/uvicorn.pid)"
    exit 0
  fi
  sleep 1
done

echo "Server failed to start. Log:"
tail -n 200 /content/uvicorn.log || true
exit 1

{"ok":true,"model_loaded":false,"mode":"raw_mesh_obj","endpoints":["/health","/generate-raw-obj","/generate-obj"]}
Server up. PID=6204


+ source /content/py310/bin/activate
++ '[' -z '' ']'
++ '[' -n x ']'
++ SCRIPT_PATH=/content/py310/bin/activate
++ '[' /content/py310/bin/activate = bash ']'
++ deactivate nondestructive
++ unset -f pydoc
++ '[' -z '' ']'
++ '[' -z '' ']'
++ hash -r
++ '[' -z '' ']'
++ unset VIRTUAL_ENV
++ unset VIRTUAL_ENV_PROMPT
++ '[' '!' nondestructive = nondestructive ']'
++ VIRTUAL_ENV=/content/py310
++ '[' linux-gnu = cygwin ']'
++ '[' linux-gnu = msys ']'
++ export VIRTUAL_ENV
++ '[' -z '' ']'
++ unset SCRIPT_PATH
++ _OLD_VIRTUAL_PATH=/opt/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tools/node/bin:/tools/google-cloud-sdk/bin
++ PATH=/content/py310/bin:/opt/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tools/node/bin:/tools/google-cloud-sdk/bin
++ export PATH
++ '[' x '!=' x ']'
+++ basename /content/py310
++ VIRTUAL_ENV_PROMPT=py310
++ export VIRTUAL_ENV_PROMPT
++ '[' -z '' ']'
++ '[' -z '' ']'
++ _OLD_VIRTUAL_PS

In [12]:
%%bash
set -euxo pipefail

TOKEN="$(tr -d '[:space:]' < /content/ngrok_authtoken.txt)"
ngrok config add-authtoken "$TOKEN"

pkill -f "ngrok http" || true

nohup ngrok http http://127.0.0.1:8000 --log=stdout > /content/ngrok.log 2>&1 &
echo $! > /content/ngrok.pid

sleep 2
echo "== ngrok public URL =="
grep -oE 'https://[a-zA-Z0-9.-]+\.ngrok-free\.(dev|app)' -m 1 /content/ngrok.log || true
tail -n 30 /content/ngrok.log

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml
== ngrok public URL ==
https://jeannetta-unreplete-dowdily.ngrok-free.dev
t=2026-03-23T17:19:03+0000 lvl=info msg="no configuration paths supplied"
t=2026-03-23T17:19:03+0000 lvl=info msg="using configuration at default config path" path=/root/.config/ngrok/ngrok.yml
t=2026-03-23T17:19:03+0000 lvl=info msg="open config file" path=/root/.config/ngrok/ngrok.yml err=nil
t=2026-03-23T17:19:03+0000 lvl=info msg="FIPS 140 mode" enabled=false
t=2026-03-23T17:19:03+0000 lvl=info msg="starting web service" obj=web addr=127.0.0.1:4040 allow_hosts=[]
t=2026-03-23T17:19:04+0000 lvl=info msg="client session established" obj=tunnels.session
t=2026-03-23T17:19:04+0000 lvl=info msg="tunnel session started" obj=tunnels.session
t=2026-03-23T17:19:04+0000 lvl=info msg="started tunnel" obj=tunnels name=command_line addr=http://127.0.0.1:8000 url=https://jeannetta-unreplete-dowdily.ngrok-free.dev


++ tr -d '[:space:]'
+ TOKEN=3AMkbjJGvvLTEvEFedqvrGJMXJZ_6SVsSVcrgRQFuxTN3F23o
+ ngrok config add-authtoken 3AMkbjJGvvLTEvEFedqvrGJMXJZ_6SVsSVcrgRQFuxTN3F23o
+ pkill -f 'ngrok http'
+ true
+ echo 6521
+ nohup ngrok http http://127.0.0.1:8000 --log=stdout
+ sleep 2
+ echo '== ngrok public URL =='
+ grep -oE 'https://[a-zA-Z0-9.-]+\.ngrok-free\.(dev|app)' -m 1 /content/ngrok.log
+ tail -n 30 /content/ngrok.log
